# LeetCode 3318 — Find X-Sum of All K-Long Subarrays I bằng Python

Link đề: https://leetcode.com/problems/find-x-sum-of-all-k-long-subarrays-i/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Sliding Window, Heap

Skeleton LeetCode:

```python
class Solution:
    def findXSum(self, nums: List[int], k: int, x: int) -> List[int]:
        ...
```

Ghi chú: đề và ví dụ được dịch từ đề gốc (Weekly Contest 419); mọi lời giải trong notebook đã được chạy và kiểm chứng bằng code thật.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: với **mỗi đoạn con liên tiếp dài `k`**, tính **x-sum** = tổng của các phần tử còn lại sau khi chỉ giữ lại các lần xuất hiện của `x` giá trị **thường xuyên nhất** (hoà số lần thì giá trị **lớn hơn** thắng).
2. Nắm các công cụ Python: `collections.Counter` (đếm tần suất), **so sánh tuple** `(tần_suất, giá_trị)`, `sorted(..., reverse=True)`, cắt lát `[:x]`, `heapq.nlargest`, và `bisect` để giữ danh sách luôn được sắp xếp.
3. Đi từ cách **tính lại từ đầu cho mỗi cửa sổ** → **cửa sổ trượt (sliding window) cập nhật bộ đếm** → **chọn top-`x` bằng heap** → **danh sách đã sắp xếp cập nhật bằng `bisect`**.
4. Hiểu mấu chốt: luật "nhiều lần hơn thắng, hoà thì giá trị lớn hơn thắng" **chính là so sánh tuple** `(số lần, giá trị)` theo thứ tự giảm dần; và hai cửa sổ liên tiếp chỉ khác nhau **đúng hai phần tử** (một vào, một ra).
5. Tránh các lỗi kinh điển: sai luật hoà, tính tổng giá trị mà quên nhân số lần, sai số cửa sổ (`n - k + 1`), và ghi kết quả khi cửa sổ có `k + 1` phần tử (bỏ phần tử bên trái trễ một nhịp).

## 2. Đề bài & ví dụ

Cho mảng `nums` gồm `n` số nguyên và hai số nguyên `k`, `x`.

**x-sum** của một mảng được tính như sau:

1. Đếm số lần xuất hiện của mọi phần tử.
2. Chỉ giữ lại **các lần xuất hiện** của `x` phần tử **thường xuyên nhất**. Nếu hai phần tử có cùng số lần thì phần tử có **giá trị lớn hơn** được coi là thường xuyên hơn.
3. Tính **tổng** của mảng còn lại.

Lưu ý: nếu mảng có **ít hơn `x` giá trị phân biệt** thì x-sum bằng tổng của cả mảng.

Trả về mảng `answer` độ dài `n - k + 1`, trong đó `answer[i]` là x-sum của đoạn con `nums[i..i + k - 1]`.

**Ví dụ 1**

- Input: `nums = [1,1,2,2,3,4,2,3]`, `k = 6`, `x = 2`
- Output: `[6,10,12]`
- Giải thích: đoạn `[1,1,2,2,3,4]` giữ `1` và `2` ⇒ `1+1+2+2 = 6`. Đoạn `[1,2,2,3,4,2]` giữ `2` (3 lần) và `4` (do `1`, `3`, `4` cùng 1 lần nên `4` lớn nhất thắng) ⇒ `2+2+2+4 = 10`. Đoạn `[2,2,3,4,2,3]` giữ `2` và `3` ⇒ `2+2+2+3+3 = 12`.

**Ví dụ 2**

- Input: `nums = [3,8,7,8,7,5]`, `k = 2`, `x = 2`
- Output: `[11,15,15,15,12]`
- Giải thích: `k == x` nên mọi giá trị đều được giữ, x-sum bằng tổng cả đoạn.

**Ràng buộc:** `1 <= n == nums.length <= 50`; `1 <= nums[i] <= 50`; `1 <= x <= k <= nums.length`.

In [1]:
import random
import heapq
import bisect
from collections import Counter
from typing import List

examples = [
    (([1, 1, 2, 2, 3, 4, 2, 3], 6, 2), [6, 10, 12]),
    (([3, 8, 7, 8, 7, 5], 2, 2), [11, 15, 15, 15, 12]),
]
for (nums, k, x), expected in examples:
    print(nums, "k =", k, "x =", x, "-> kỳ vọng", expected)

[1, 1, 2, 2, 3, 4, 2, 3] k = 6 x = 2 -> kỳ vọng [6, 10, 12]
[3, 8, 7, 8, 7, 5] k = 2 x = 2 -> kỳ vọng [11, 15, 15, 15, 12]


## 3. Kiến thức nền

### 3.1. Đếm tần suất bằng `Counter`

`Counter(iterable)` trả về một **dict** đặc biệt: khoá là phần tử, giá trị là số lần xuất hiện. Truy cập khoá chưa có cho `0` (không báo lỗi). `.items()` cho các cặp `(phần_tử, số_lần)`.

In [2]:
from collections import Counter

cnt = Counter([1, 1, 2, 2, 3, 4])
print(cnt)                               # Counter({1: 2, 2: 2, 3: 1, 4: 1})
print(cnt[2], cnt[99])                   # 2 0   (khoá chưa có -> 0)
print(list(cnt.items()))                 # [(1, 2), (2, 2), (3, 1), (4, 1)]
cnt[3] += 1                              # tăng một lần đếm
print(cnt[3], len(cnt))                  # 2 4   (len = số giá trị phân biệt)

Counter({1: 2, 2: 2, 3: 1, 4: 1})
2 0
[(1, 2), (2, 2), (3, 1), (4, 1)]
2 4


### 3.2. So sánh tuple: so từng phần từ trái sang phải

Python so sánh hai tuple bằng cách so **phần tử đầu**; nếu bằng nhau mới so phần tử kế. Đúng bằng luật của đề nếu ta ghép **`(số_lần, giá_trị)`**:

- `(2, 1)` so với `(1, 9)`: `2 > 1` ⇒ cặp thứ nhất lớn hơn (nhiều lần hơn thắng).
- `(1, 4)` so với `(1, 3)`: số lần bằng nhau, `4 > 3` ⇒ cặp thứ nhất lớn hơn (hoà thì giá trị lớn hơn thắng).

In [3]:
print((2, 1) > (1, 9))                   # True   nhiều lần hơn thắng
print((1, 4) > (1, 3))                   # True   hoà số lần -> giá trị lớn hơn thắng
print(max([(1, 3), (1, 4), (1, 2)]))     # (1, 4)

True
True
(1, 4)


### 3.3. `sorted(..., reverse=True)` và cắt lát `[:x]`

`sorted(list, reverse=True)` trả về list mới sắp **giảm dần**. Với list các tuple, nó sắp theo quy tắc so sánh tuple ở trên. `lst[:x]` lấy `x` phần tử **đầu** (nếu list ngắn hơn `x` thì lấy hết, không báo lỗi). Như vậy "top `x`" = `sorted(pairs, reverse=True)[:x]`.

In [4]:
pairs = [(2, 1), (2, 2), (1, 3), (1, 4)]
top = sorted(pairs, reverse=True)
print(top)                               # [(2, 2), (2, 1), (1, 4), (1, 3)]
print(top[:2])                           # [(2, 2), (2, 1)]
print(top[:10])                          # lấy nhiều hơn độ dài -> lấy hết, không lỗi
print(sum(c * v for c, v in top[:2]))    # 2*2 + 2*1 = 6

[(2, 2), (2, 1), (1, 4), (1, 3)]
[(2, 2), (2, 1)]
[(2, 2), (2, 1), (1, 4), (1, 3)]
6


### 3.4. Cửa sổ trượt kích thước `k`

Có `n` phần tử thì số đoạn con liên tiếp dài `k` là **`n - k + 1`** (bắt đầu tại `i = 0, 1, ..., n - k`). Đoạn thứ `i` là `nums[i : i + k]` (cắt lát lấy `k` phần tử, **không** gồm chỉ số `i + k`).

In [5]:
nums = [1, 1, 2, 2, 3, 4, 2, 3]
n, k = len(nums), 6
print("số cửa sổ:", n - k + 1)           # 3
for i in range(n - k + 1):
    print(i, nums[i:i + k])

số cửa sổ: 3
0 [1, 1, 2, 2, 3, 4]
1 [1, 2, 2, 3, 4, 2]
2 [2, 2, 3, 4, 2, 3]


### 3.5. `heapq.nlargest(x, iterable)`

`heapq.nlargest(x, iterable)` trả về list `x` phần tử **lớn nhất** (giảm dần) mà không phải sắp xếp cả dãy; với tuple nó cũng dùng so sánh tuple. Nếu có ít hơn `x` phần tử thì trả hết.

In [6]:
pairs = [(2, 1), (2, 2), (1, 3), (1, 4)]
print(heapq.nlargest(2, pairs))          # [(2, 2), (2, 1)]
print(heapq.nlargest(9, pairs))          # trả hết, giảm dần

[(2, 2), (2, 1)]
[(2, 2), (2, 1), (1, 4), (1, 3)]


### 3.6. `bisect`: giữ list luôn được sắp xếp

Nếu `lst` đã tăng dần thì `bisect.insort(lst, item)` chèn `item` đúng chỗ để `lst` vẫn tăng dần, còn `bisect.bisect_left(lst, item)` trả về vị trí của `item` (để xoá bằng `del lst[pos]`). Dùng để cập nhật danh sách top mà **không sắp xếp lại từ đầu**.

In [7]:
lst = [(1, 3), (2, 1)]
bisect.insort(lst, (1, 4))
print(lst)                               # [(1, 3), (1, 4), (2, 1)]
pos = bisect.bisect_left(lst, (1, 4))
print(pos, lst[pos])                     # 1 (1, 4)
del lst[pos]
print(lst)                               # [(1, 3), (2, 1)]

[(1, 3), (1, 4), (2, 1)]
1 (1, 4)
[(1, 3), (2, 1)]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Tính x-sum của đoạn đầu ví dụ 1: `[1, 1, 2, 2, 3, 4]`, `x = 2`.

1. Đếm: `1 → 2 lần`, `2 → 2 lần`, `3 → 1 lần`, `4 → 1 lần`.
2. Ghép thành cặp `(số_lần, giá_trị)`: `(2,1)`, `(2,2)`, `(1,3)`, `(1,4)`.
3. Xếp giảm dần: `(2,2)`, `(2,1)`, `(1,4)`, `(1,3)`.
4. Lấy `x = 2` cặp đầu: `(2,2)` và `(2,1)`.
5. Mỗi cặp đóng góp `số_lần × giá_trị`: `2×2 + 2×1 = 6`.

Chú ý bước 5: ta giữ **mọi lần xuất hiện** của giá trị đã chọn (nên phải nhân với số lần), không chỉ một lần.

In [8]:
window = [1, 1, 2, 2, 3, 4]
x = 2
cnt = Counter(window)
pairs = [(c, v) for v, c in cnt.items()]
print("cặp (số_lần, giá_trị):", pairs)
top = sorted(pairs, reverse=True)[:x]
print("top", x, ":", top)
print("x-sum =", sum(c * v for c, v in top))     # 6

cặp (số_lần, giá_trị): [(2, 1), (2, 2), (1, 3), (1, 4)]
top 2 : [(2, 2), (2, 1)]
x-sum = 6


## 5. Cách 1 — Brute Force / baseline: tính lại từ đầu cho mỗi cửa sổ

**Ý tưởng.** Với mỗi `i` từ `0` đến `n - k`: cắt `window = nums[i:i+k]`, đếm bằng `Counter`, tạo các cặp, sắp xếp giảm dần, lấy `x` cặp đầu và cộng `số_lần × giá_trị`. Đúng như mục 4, làm lặp lại cho từng cửa sổ.

In [9]:
def find_xsum_brute(nums: List[int], k: int, x: int) -> List[int]:
    n = len(nums)
    ans = []
    for i in range(n - k + 1):
        window = nums[i:i + k]                                   # cắt cửa sổ
        cnt = Counter(window)                                    # đếm tần suất
        pairs = sorted(((c, v) for v, c in cnt.items()), reverse=True)
        ans.append(sum(c * v for c, v in pairs[:x]))             # chỉ giữ top x
    return ans


for (nums, k, x), expected in examples:
    print(find_xsum_brute(nums, k, x), "(kỳ vọng", expected, ")")

[6, 10, 12] (kỳ vọng [6, 10, 12] )
[11, 15, 15, 15, 12] (kỳ vọng [11, 15, 15, 15, 12] )


**Độ phức tạp (Cách 1).** Có `n - k + 1` cửa sổ. Mỗi cửa sổ: cắt `O(k)`, đếm `O(k)`, sắp xếp tối đa `d` cặp (`d <=` số giá trị phân biệt `<= min(k, V)`) tốn `O(d log d)`. ⇒ **Time `O((n - k + 1) · (k + d log d))`**, tức tối đa `O(n · k log k)`; **Space `O(k)`**.

**Nút thắt:** hai cửa sổ liên tiếp chia sẻ `k - 1` phần tử, nhưng ta **đếm lại toàn bộ** mỗi lần. Ta chỉ cần cập nhật bộ đếm khi một phần tử vào và một phần tử ra.

## 6. Ý tưởng thuật toán / bước đột phá

Khi cửa sổ trượt sang phải một bước:

- phần tử mới `nums[i]` **vào** ⇒ `cnt[nums[i]] += 1`;
- phần tử cũ `nums[i - k]` **ra** ⇒ `cnt[nums[i - k]] -= 1` (nếu về `0` thì xoá khỏi bộ đếm).

Như vậy bộ đếm luôn đúng cho cửa sổ hiện tại mà **chỉ tốn `O(1)` mỗi lần cập nhật**. Việc còn lại là mỗi cửa sổ chọn top `x` trong **các giá trị phân biệt** (tối đa `d <= 50`), thay vì cả `k` phần tử.

In [10]:
nums = [1, 1, 2, 2, 3, 4, 2, 3]
k = 6
cnt = Counter(nums[:k])                  # cửa sổ đầu tiên
print("cửa sổ 0:", dict(cnt))
for i in range(k, len(nums)):
    cnt[nums[i]] += 1                    # phần tử mới vào
    out = nums[i - k]                    # phần tử cũ ra
    cnt[out] -= 1
    if cnt[out] == 0:
        del cnt[out]                     # bỏ các khoá có số lần 0
    print(f"vào {nums[i]}, ra {out} -> cửa sổ {i - k + 1}:", dict(cnt))

cửa sổ 0: {1: 2, 2: 2, 3: 1, 4: 1}
vào 2, ra 1 -> cửa sổ 1: {1: 1, 2: 3, 3: 1, 4: 1}
vào 3, ra 1 -> cửa sổ 2: {2: 3, 3: 2, 4: 1}


## 7. Các cách tiếp theo

### Cách 2 — Cửa sổ trượt + sắp xếp các giá trị phân biệt

**Giải thích từng đoạn.**

1. `cnt = Counter()`: bộ đếm rỗng; `ans = []`.
2. `for i, v in enumerate(nums)`: duyệt từng phần tử `v` và đưa vào cửa sổ (`cnt[v] += 1`).
3. `left = i - k`: nếu `left >= 0` nghĩa là cửa sổ đã dài `k + 1` nên phải **bỏ** `nums[left]`: giảm đếm, về `0` thì `del`.
4. `if i >= k - 1`: cửa sổ đủ `k` phần tử ⇒ tính x-sum từ `cnt` bằng cách sắp xếp các cặp `(số_lần, giá_trị)`.

In [11]:
def x_sum_from_counter(cnt: Counter, x: int) -> int:
    pairs = sorted(((c, v) for v, c in cnt.items()), reverse=True)
    return sum(c * v for c, v in pairs[:x])


def find_xsum_sliding_sort(nums: List[int], k: int, x: int) -> List[int]:
    cnt = Counter()
    ans = []
    for i, v in enumerate(nums):
        cnt[v] += 1                      # phần tử mới vào cửa sổ
        left = i - k
        if left >= 0:                    # cửa sổ đang có k + 1 phần tử -> bỏ phần tử trái nhất
            cnt[nums[left]] -= 1
            if cnt[nums[left]] == 0:
                del cnt[nums[left]]
        if i >= k - 1:                   # cửa sổ đủ k phần tử
            ans.append(x_sum_from_counter(cnt, x))
    return ans


for (nums, k, x), expected in examples:
    print(find_xsum_sliding_sort(nums, k, x), "(kỳ vọng", expected, ")")

[6, 10, 12] (kỳ vọng [6, 10, 12] )
[11, 15, 15, 15, 12] (kỳ vọng [11, 15, 15, 15, 12] )


In [12]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionSlidingSort:
    def findXSum(self, nums: List[int], k: int, x: int) -> List[int]:
        cnt = Counter()
        ans = []
        for i, v in enumerate(nums):
            cnt[v] += 1
            left = i - k
            if left >= 0:
                cnt[nums[left]] -= 1
                if cnt[nums[left]] == 0:
                    del cnt[nums[left]]
            if i >= k - 1:
                pairs = sorted(((c, val) for val, c in cnt.items()), reverse=True)
                ans.append(sum(c * val for c, val in pairs[:x]))
        return ans


print(SolutionSlidingSort().findXSum([3, 8, 7, 8, 7, 5], 2, 2))   # [11, 15, 15, 15, 12]

[11, 15, 15, 15, 12]


**Độ phức tạp Cách 2:** cập nhật bộ đếm `O(1)` mỗi bước; mỗi cửa sổ sắp xếp `d` cặp (`d <= min(k, 50)`) tốn `O(d log d)` ⇒ **Time `O(n · d log d)`**, **Space `O(d)`** cho bộ đếm (cộng `O(d)` list tạm).

### Cách 3 — Cửa sổ trượt + `heapq.nlargest`

**Giải thích.** Ta chỉ cần **top `x`**, không cần sắp xếp toàn bộ `d` cặp. `heapq.nlargest(x, pairs)` giữ một heap kích thước `x` nên mất `O(d log x)`. Phần cửa sổ trượt giữ y hệt Cách 2; chỉ thay dòng tính x-sum.

In [13]:
def find_xsum_heap(nums: List[int], k: int, x: int) -> List[int]:
    cnt = Counter()
    ans = []
    for i, v in enumerate(nums):
        cnt[v] += 1
        left = i - k
        if left >= 0:
            cnt[nums[left]] -= 1
            if cnt[nums[left]] == 0:
                del cnt[nums[left]]
        if i >= k - 1:
            top = heapq.nlargest(x, ((c, val) for val, c in cnt.items()))   # top x theo (số_lần, giá_trị)
            ans.append(sum(c * val for c, val in top))
    return ans


for (nums, k, x), expected in examples:
    print(find_xsum_heap(nums, k, x), "(kỳ vọng", expected, ")")

[6, 10, 12] (kỳ vọng [6, 10, 12] )
[11, 15, 15, 15, 12] (kỳ vọng [11, 15, 15, 15, 12] )


In [14]:
# Code nộp LeetCode cho Cách 3 (đổi tên lớp thành Solution khi nộp)
class SolutionHeap:
    def findXSum(self, nums: List[int], k: int, x: int) -> List[int]:
        cnt = Counter()
        ans = []
        for i, v in enumerate(nums):
            cnt[v] += 1
            if i >= k:
                cnt[nums[i - k]] -= 1
                if cnt[nums[i - k]] == 0:
                    del cnt[nums[i - k]]
            if i >= k - 1:
                top = heapq.nlargest(x, ((c, val) for val, c in cnt.items()))
                ans.append(sum(c * val for c, val in top))
        return ans


print(SolutionHeap().findXSum([1, 1, 2, 2, 3, 4, 2, 3], 6, 2))   # [6, 10, 12]

[6, 10, 12]


**Độ phức tạp Cách 3:** **Time `O(n · d log x)`**, **Space `O(d + x)`**.

### Cách 4 — Danh sách `(số_lần, giá_trị)` luôn được sắp xếp, cập nhật bằng `bisect`

**Giải thích từng đoạn.** Ý tưởng của lời giải chính thức (hai tập có thứ tự) được thu gọn ở đây thành **một list tăng dần** `ordered` chứa các cặp `(số_lần, giá_trị)` của các giá trị đang có trong cửa sổ:

1. Khi số lần của `v` đổi từ `old` sang `new`: **xoá** cặp cũ `(old, v)` (nếu `old > 0`) bằng `bisect_left` + `del`, cập nhật `cnt[v]`, rồi **chèn** cặp mới `(new, v)` (nếu `new > 0`) bằng `insort`. Thứ tự: xoá cũ → đổi đếm → chèn mới, để list không bao giờ có cặp "ma".
2. List tăng dần nên **`x` phần tử cuối** là top `x`. x-sum = `sum(c * v for c, v in ordered[-x:])`. Nếu `len(ordered) < x`, `ordered[-x:]` tự lấy hết list (đúng quy tắc "ít hơn `x` giá trị thì cộng tất cả").

In [15]:
def find_xsum_bisect(nums: List[int], k: int, x: int) -> List[int]:
    cnt = {}                             # giá trị -> số lần trong cửa sổ
    ordered = []                         # các cặp (số_lần, giá_trị), luôn tăng dần

    def change(v: int, delta: int) -> None:
        old = cnt.get(v, 0)
        if old > 0:                      # bỏ cặp cũ khỏi list có thứ tự
            pos = bisect.bisect_left(ordered, (old, v))
            del ordered[pos]
        new = old + delta
        if new > 0:
            cnt[v] = new
            bisect.insort(ordered, (new, v))         # chèn cặp mới đúng chỗ
        else:
            cnt.pop(v, None)

    ans = []
    for i, v in enumerate(nums):
        change(v, +1)
        if i >= k:
            change(nums[i - k], -1)
        if i >= k - 1:
            ans.append(sum(c * val for c, val in ordered[-x:]))
    return ans


for (nums, k, x), expected in examples:
    print(find_xsum_bisect(nums, k, x), "(kỳ vọng", expected, ")")

[6, 10, 12] (kỳ vọng [6, 10, 12] )
[11, 15, 15, 15, 12] (kỳ vọng [11, 15, 15, 15, 12] )


In [16]:
# Code nộp LeetCode cho Cách 4 (đổi tên lớp thành Solution khi nộp)
class SolutionBisect:
    def findXSum(self, nums: List[int], k: int, x: int) -> List[int]:
        import bisect
        cnt, ordered, ans = {}, [], []

        def change(v, delta):
            old = cnt.get(v, 0)
            if old > 0:
                del ordered[bisect.bisect_left(ordered, (old, v))]
            new = old + delta
            if new > 0:
                cnt[v] = new
                bisect.insort(ordered, (new, v))
            else:
                cnt.pop(v, None)

        for i, v in enumerate(nums):
            change(v, 1)
            if i >= k:
                change(nums[i - k], -1)
            if i >= k - 1:
                ans.append(sum(c * val for c, val in ordered[-x:]))
        return ans


print(SolutionBisect().findXSum([3, 8, 7, 8, 7, 5], 2, 2))   # [11, 15, 15, 15, 12]

[11, 15, 15, 15, 12]


**Độ phức tạp Cách 4:** mỗi lần `change` tốn `O(d)` (tìm bằng `bisect` là `O(log d)` nhưng `del`/`insort` dịch phần tử trong list là `O(d)`), tính tổng `O(x)` ⇒ **Time `O(n · (d + x))`**, **Space `O(d)`**. Với `d <= 50` rất nhanh; với dữ liệu lớn (bài **II**) người ta dùng cấu trúc có thứ tự thực sự (`SortedList`, hai heap lười xoá) để mỗi cập nhật chỉ `O(log d)`.

## 8. TRACE từng bước

**Bảng trace Cách 2** với `nums = [1,1,2,2,3,4,2,3]`, `k = 6`, `x = 2` (ghi tại thời điểm `i >= k - 1`):

| i | vào | ra | `cnt` sau khi cập nhật | cặp sắp giảm dần | top 2 | x-sum |
|---|---|---|---|---|---|---|
| 5 | 4 | — | `{1:2, 2:2, 3:1, 4:1}` | `(2,2) (2,1) (1,4) (1,3)` | `(2,2) (2,1)` | 6 |
| 6 | 2 | `nums[0]=1` | `{1:1, 2:3, 3:1, 4:1}` | `(3,2) (1,4) (1,3) (1,1)` | `(3,2) (1,4)` | 10 |
| 7 | 3 | `nums[1]=1` | `{2:3, 3:2, 4:1}` | `(3,2) (2,3) (1,4)` | `(3,2) (2,3)` | 12 |

Ở dòng `i = 6`: `1`, `3`, `4` cùng 1 lần, nhưng `4` lớn nhất nên được chọn cùng `2`; đó là điểm cần chú ý của luật hoà.

In [17]:
def trace_sliding(nums: List[int], k: int, x: int) -> List[int]:
    cnt = Counter()
    ans = []
    print(f"nums={nums}, k={k}, x={x}")
    for i, v in enumerate(nums):
        cnt[v] += 1
        out = None
        left = i - k
        if left >= 0:
            out = nums[left]
            cnt[out] -= 1
            if cnt[out] == 0:
                del cnt[out]
        print(f"  i={i}: vào {v}, ra {out}, cnt={dict(sorted(cnt.items()))}")
        if i >= k - 1:
            pairs = sorted(((c, val) for val, c in cnt.items()), reverse=True)
            top = pairs[:x]
            total = sum(c * val for c, val in top)
            print(f"      cặp giảm dần={pairs}")
            print(f"      top {x}={top} -> x-sum = " + " + ".join(f"{c}*{val}" for c, val in top) + f" = {total}")
            ans.append(total)
    print("  answer =", ans)
    return ans


assert trace_sliding([1, 1, 2, 2, 3, 4, 2, 3], 6, 2) == [6, 10, 12]

nums=[1, 1, 2, 2, 3, 4, 2, 3], k=6, x=2
  i=0: vào 1, ra None, cnt={1: 1}
  i=1: vào 1, ra None, cnt={1: 2}
  i=2: vào 2, ra None, cnt={1: 2, 2: 1}
  i=3: vào 2, ra None, cnt={1: 2, 2: 2}
  i=4: vào 3, ra None, cnt={1: 2, 2: 2, 3: 1}
  i=5: vào 4, ra None, cnt={1: 2, 2: 2, 3: 1, 4: 1}
      cặp giảm dần=[(2, 2), (2, 1), (1, 4), (1, 3)]
      top 2=[(2, 2), (2, 1)] -> x-sum = 2*2 + 2*1 = 6
  i=6: vào 2, ra 1, cnt={1: 1, 2: 3, 3: 1, 4: 1}
      cặp giảm dần=[(3, 2), (1, 4), (1, 3), (1, 1)]
      top 2=[(3, 2), (1, 4)] -> x-sum = 3*2 + 1*4 = 10
  i=7: vào 3, ra 1, cnt={2: 3, 3: 2, 4: 1}
      cặp giảm dần=[(3, 2), (2, 3), (1, 4)]
      top 2=[(3, 2), (2, 3)] -> x-sum = 3*2 + 2*3 = 12
  answer = [6, 10, 12]


In [18]:
def trace_bisect(nums: List[int], k: int, x: int) -> List[int]:
    cnt, ordered, ans = {}, [], []
    print(f"nums={nums}, k={k}, x={x}")

    def change(v, delta):
        old = cnt.get(v, 0)
        if old > 0:
            del ordered[bisect.bisect_left(ordered, (old, v))]
        new = old + delta
        if new > 0:
            cnt[v] = new
            bisect.insort(ordered, (new, v))
        else:
            cnt.pop(v, None)
        print(f"    giá trị {v} {delta:+d}: số lần {old} -> {new}, ordered={ordered}")

    for i, v in enumerate(nums):
        print(f"  i={i}")
        change(v, +1)
        if i >= k:
            change(nums[i - k], -1)
        if i >= k - 1:
            top = ordered[-x:]
            total = sum(c * val for c, val in top)
            print(f"    {x} cặp cuối={top} -> x-sum={total}")
            ans.append(total)
    return ans


assert trace_bisect([3, 8, 7, 8, 7, 5], 2, 2) == [11, 15, 15, 15, 12]

nums=[3, 8, 7, 8, 7, 5], k=2, x=2
  i=0
    giá trị 3 +1: số lần 0 -> 1, ordered=[(1, 3)]
  i=1
    giá trị 8 +1: số lần 0 -> 1, ordered=[(1, 3), (1, 8)]
    2 cặp cuối=[(1, 3), (1, 8)] -> x-sum=11
  i=2
    giá trị 7 +1: số lần 0 -> 1, ordered=[(1, 3), (1, 7), (1, 8)]
    giá trị 3 -1: số lần 1 -> 0, ordered=[(1, 7), (1, 8)]
    2 cặp cuối=[(1, 7), (1, 8)] -> x-sum=15
  i=3
    giá trị 8 +1: số lần 1 -> 2, ordered=[(1, 7), (2, 8)]
    giá trị 8 -1: số lần 2 -> 1, ordered=[(1, 7), (1, 8)]
    2 cặp cuối=[(1, 7), (1, 8)] -> x-sum=15
  i=4
    giá trị 7 +1: số lần 1 -> 2, ordered=[(1, 8), (2, 7)]
    giá trị 7 -1: số lần 2 -> 1, ordered=[(1, 7), (1, 8)]
    2 cặp cuối=[(1, 7), (1, 8)] -> x-sum=15
  i=5
    giá trị 5 +1: số lần 0 -> 1, ordered=[(1, 5), (1, 7), (1, 8)]
    giá trị 8 -1: số lần 1 -> 0, ordered=[(1, 5), (1, 7)]
    2 cặp cuối=[(1, 5), (1, 7)] -> x-sum=12


## 9. Code nộp LeetCode

Ta nộp **Cách 2** (cửa sổ trượt + sắp xếp các giá trị phân biệt): ngắn, dễ đọc, và độ phức tạp thừa đủ cho `n <= 50`.

In [19]:
class Solution:
    def findXSum(self, nums: List[int], k: int, x: int) -> List[int]:
        cnt = Counter()
        ans = []
        for i, v in enumerate(nums):
            cnt[v] += 1                              # phần tử mới vào cửa sổ
            left = i - k
            if left >= 0:                            # bỏ phần tử vừa ra khỏi cửa sổ
                cnt[nums[left]] -= 1
                if cnt[nums[left]] == 0:
                    del cnt[nums[left]]
            if i >= k - 1:                           # cửa sổ đủ k phần tử
                pairs = sorted(((c, val) for val, c in cnt.items()), reverse=True)
                ans.append(sum(c * val for c, val in pairs[:x]))
        return ans


sol = Solution()
print(sol.findXSum([1, 1, 2, 2, 3, 4, 2, 3], 6, 2))   # [6, 10, 12]
print(sol.findXSum([3, 8, 7, 8, 7, 5], 2, 2))         # [11, 15, 15, 15, 12]

[6, 10, 12]
[11, 15, 15, 15, 12]


## 10. Vì sao đúng

- **Bộ đếm luôn khớp cửa sổ.** Sau khi xử lý chỉ số `i`, bộ đếm chứa các phần tử `nums[max(0, i - k + 1) .. i]`: ta thêm `nums[i]` và bỏ `nums[i - k]` (phần tử vừa trượt ra khỏi cửa sổ độ dài `k`). Khi `i >= k - 1`, cửa sổ có đúng `k` phần tử `nums[i - k + 1 .. i]`, tức đoạn thứ `i - k + 1`; ta ghi đúng `n - k + 1` kết quả theo thứ tự.
- **Tuple `(số_lần, giá_trị)` mã hoá đúng luật xếp hạng.** So sánh tuple xét số lần trước (nhiều hơn là lớn hơn), nếu bằng thì xét giá trị (lớn hơn là lớn hơn) — đúng luật đề; các cặp có khoá phân biệt (vì giá trị khác nhau) nên thứ tự là **toàn phần**, không còn hoà.
- **Top `x` và tổng.** Các giá trị trong `pairs[:x]` là `x` giá trị xếp hạng cao nhất (hoặc tất cả nếu ít hơn `x`, vì cắt lát an toàn). "Giữ lại mọi lần xuất hiện của chúng" nghĩa là mỗi giá trị `v` đóng góp `cnt[v] × v` vào tổng.
- Cách 3 và Cách 4 chỉ thay cách lấy top `x` bằng một cấu trúc khác nhưng **cùng** thứ tự so sánh nên cho cùng kết quả.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `d` = số giá trị phân biệt trong một cửa sổ (`d <= min(k, 50)`).

| Cách | Time | Space |
|---|---|---|
| 1. Tính lại mỗi cửa sổ | `O(n · (k + d log d))` | `O(k)` |
| 2. Trượt + sắp xếp | `O(n · d log d)` | `O(d)` |
| 3. Trượt + `heapq.nlargest` | `O(n · d log x)` | `O(d + x)` |
| 4. Trượt + list có thứ tự (`bisect`) | `O(n · (d + x))` | `O(d)` |

Với `n <= 50` mọi cách đều tức thì. Ý nghĩa thực tế: bài II (`n` lớn) cần cập nhật `O(log d)` mỗi bước nên ta cần cấu trúc có thứ tự đúng nghĩa.

## 12. Bộ test

Để đối chiếu **độc lập**, ta viết `x_sum_reference(window, x)` làm đúng từng chữ của đề: xếp hạng các giá trị, giữ lại các lần xuất hiện của `x` giá trị đầu **bằng cách dựng mảng mới**, rồi cộng. Hàm này không dùng tuple `(số_lần, giá_trị)` mà dùng `key` riêng.

In [20]:
def x_sum_reference(window, x):
    counts = {}
    for v in window:
        counts[v] = counts.get(v, 0) + 1
    # xếp hạng: nhiều lần hơn trước; hoà thì giá trị lớn hơn trước
    ranked = sorted(counts, key=lambda v: (-counts[v], -v))
    kept_values = set(ranked[:x])
    kept_array = [v for v in window if v in kept_values]   # chỉ giữ các lần xuất hiện của top x
    return sum(kept_array)


def reference(nums, k, x):
    return [x_sum_reference(nums[i:i + k], x) for i in range(len(nums) - k + 1)]


impls = {
    "brute": find_xsum_brute,
    "sliding_sort": find_xsum_sliding_sort,
    "heap": find_xsum_heap,
    "bisect": find_xsum_bisect,
    "leetcode": Solution().findXSum,
}


def check_all(nums, k, x, label, expected=None):
    ref = reference(nums, k, x)
    if expected is not None:
        assert ref == expected, (label, ref, expected)
    for name, fn in impls.items():
        actual = fn(list(nums), k, x)
        assert actual == ref, (label, name, nums, k, x, actual, ref)
    print("PASS", label)


for idx, ((nums, k, x), expected) in enumerate(examples, 1):
    check_all(nums, k, x, f"ví dụ {idx}", expected)

PASS ví dụ 1
PASS ví dụ 2


### 12.1. Các trường hợp biên

Gồm: `k = n` (chỉ một cửa sổ), `k = 1` (mỗi cửa sổ một phần tử), `x = 1`, `x = k` (cộng cả đoạn), mảng toàn phần tử giống nhau, mảng toàn giá trị khác nhau, và trường hợp **ít hơn `x` giá trị phân biệt**.

In [21]:
check_all([5], 1, 1, "n = 1", [5])
check_all([1, 2, 3], 3, 1, "k = n, x = 1", [3])
check_all([1, 2, 3], 1, 1, "k = 1", [1, 2, 3])
check_all([5, 5, 5, 5], 4, 3, "ít hơn x giá trị phân biệt (cộng cả đoạn)", [20])
check_all([7, 7, 7, 7, 7], 3, 2, "toàn phần tử giống nhau", [21, 21, 21])
check_all([1, 2, 3, 4, 5, 6], 4, 2, "toàn giá trị khác nhau: chọn 2 giá trị lớn nhất", [7, 9, 11])
check_all([1, 2, 2, 3, 3, 3], 4, 1, "x = 1", [4, 6, 9])
check_all([3, 8, 7, 8, 7, 5], 6, 6, "x = k = n")
check_all([50] * 50, 50, 50, "giá trị lớn nhất, n = 50", [2500])
check_all(list(range(1, 51)), 50, 1, "50 giá trị khác nhau, x = 1", [50])
check_all(list(range(1, 51)), 25, 25, "x = k, tổng cả cửa sổ")

PASS n = 1
PASS k = n, x = 1
PASS k = 1
PASS ít hơn x giá trị phân biệt (cộng cả đoạn)
PASS toàn phần tử giống nhau
PASS toàn giá trị khác nhau: chọn 2 giá trị lớn nhất
PASS x = 1
PASS x = k = n
PASS giá trị lớn nhất, n = 50
PASS 50 giá trị khác nhau, x = 1
PASS x = k, tổng cả cửa sổ


### 12.2. Phản ví dụ cho các cách làm sai

Ba lỗi: (a) xếp hạng chỉ theo số lần, hoà thì lấy theo thứ tự xuất hiện; (b) cộng mỗi giá trị **một lần** thay vì `số_lần × giá_trị`; (c) dùng `range(n - k)` bỏ mất cửa sổ cuối.

In [22]:
def wrong_tie(nums, k, x):
    ans = []
    for i in range(len(nums) - k + 1):
        cnt = Counter(nums[i:i + k])
        top = sorted(cnt.items(), key=lambda p: -p[1])[:x]     # SAI: hoà thì giữ thứ tự xuất hiện
        ans.append(sum(c * v for v, c in top))
    return ans


def wrong_no_multiply(nums, k, x):
    ans = []
    for i in range(len(nums) - k + 1):
        cnt = Counter(nums[i:i + k])
        top = sorted(((c, v) for v, c in cnt.items()), reverse=True)[:x]
        ans.append(sum(v for c, v in top))                     # SAI: thiếu nhân số lần
    return ans


def wrong_window_count(nums, k, x):
    ans = []
    for i in range(len(nums) - k):                             # SAI: thiếu cửa sổ cuối
        cnt = Counter(nums[i:i + k])
        top = sorted(((c, v) for v, c in cnt.items()), reverse=True)[:x]
        ans.append(sum(c * v for c, v in top))
    return ans


args = ([1, 1, 2, 2, 3, 4, 2, 3], 6, 2)
good = Solution().findXSum(*args)
for fn in (wrong_tie, wrong_no_multiply, wrong_window_count):
    out = fn(*args)
    print(fn.__name__, out, "!=", good)
    assert out != good

wrong_tie [6, 7, 12] != [6, 10, 12]
wrong_no_multiply [3, 6, 5] != [6, 10, 12]
wrong_window_count [6, 10] != [6, 10, 12]


### 12.3. Đối chiếu ngẫu nhiên (có seed) đúng ràng buộc

Sinh `n` trong `[1, 50]`, giá trị trong `[1, 50]`, `1 <= x <= k <= n`. Để tạo **nhiều trường hợp hoà** ta thay đổi cỡ miền giá trị (`vmax` nhỏ: 2, 3, 5, 10 hoặc 50). Mọi cách phải trùng với `reference`.

In [23]:
rng = random.Random(3318)
total = 0
for trial in range(3000):
    n = rng.randint(1, 50)
    vmax = rng.choice([2, 3, 5, 10, 50])
    nums = [rng.randint(1, vmax) for _ in range(n)]
    k = rng.randint(1, n)
    x = rng.randint(1, k)
    assert 1 <= n <= 50 and all(1 <= v <= 50 for v in nums) and 1 <= x <= k <= n
    ref = reference(nums, k, x)
    for name, fn in impls.items():
        assert fn(list(nums), k, x) == ref, (name, nums, k, x)
    total += 1
print("PASS", total, "bộ ngẫu nhiên hợp lệ")

PASS 3000 bộ ngẫu nhiên hợp lệ


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Sai luật hoà (không ưu tiên giá trị lớn hơn)

In [24]:
def sai_luat_hoa(window, x):
    cnt = Counter(window)
    top = sorted(cnt.items(), key=lambda p: -p[1])[:x]         # SAI: hoà thì theo thứ tự xuất hiện
    return sum(c * v for v, c in top)


w = [1, 2, 2, 3, 4, 2]
print("sai:", sai_luat_hoa(w, 2), "| đúng:", x_sum_reference(w, 2))    # 7 | 10

sai: 7 | đúng: 10


**Vì sao sai:** `1, 3, 4` cùng xuất hiện 1 lần, đề bắt chọn `4` (lớn nhất). `sorted` **ổn định** nên giữ nguyên thứ tự xuất hiện (`1` đứng trước) ⇒ chọn nhầm `1` (kết quả `2×3 + 1 = 7` thay vì `10`). **Cách sửa:** xếp theo **cả hai** khoá: `(số_lần, giá_trị)` giảm dần, hoặc `key=lambda p: (-p[1], -p[0])`.

### Lỗi 2: Quên nhân số lần xuất hiện

In [25]:
def sai_quen_nhan(window, x):
    cnt = Counter(window)
    top = sorted(((c, v) for v, c in cnt.items()), reverse=True)[:x]
    return sum(v for c, v in top)                              # SAI: chỉ cộng giá trị, không nhân số lần


w = [1, 1, 2, 2, 3, 4]
print("sai:", sai_quen_nhan(w, 2), "| đúng:", x_sum_reference(w, 2))    # 3 | 6

sai: 3 | đúng: 6


**Vì sao sai:** x-sum cộng **mảng còn lại**, trong đó mỗi giá trị được giữ vẫn xuất hiện đúng số lần của nó. Giá trị `2` xuất hiện 2 lần đóng góp `4`, không phải `2`. **Cách sửa:** cộng `c * v`.

### Lỗi 3: Sai số cửa sổ (`n - k` thay vì `n - k + 1`)

In [26]:
def sai_so_cua_so(nums, k, x):
    res = []
    for i in range(len(nums) - k):                             # SAI: phải là len(nums) - k + 1
        res.append(x_sum_reference(nums[i:i + k], x))
    return res


print(sai_so_cua_so([3, 8, 7, 8, 7, 5], 2, 2), "| đúng:", Solution().findXSum([3, 8, 7, 8, 7, 5], 2, 2))

[11, 15, 15, 15] | đúng: [11, 15, 15, 15, 12]


**Vì sao sai:** có `n` phần tử, đoạn dài `k` có thể bắt đầu tại `0..n-k` — tức `n - k + 1` vị trí. `range(n - k)` bỏ vị trí cuối (`12` bị mất). Kiểm tra nhanh: nếu `k = n` thì phải có đúng **1** cửa sổ. **Cách sửa:** `range(n - k + 1)`.

### Lỗi 4: Ghi kết quả khi cửa sổ có `k + 1` phần tử (bỏ phần tử trái trễ một nhịp)

In [27]:
def sai_bo_tre(nums, k, x):
    cnt = Counter()
    ans = []
    for i, v in enumerate(nums):
        cnt[v] += 1
        if i >= k - 1:
            pairs = sorted(((c, val) for val, c in cnt.items() if c > 0), reverse=True)
            ans.append(sum(c * val for c, val in pairs[:x]))   # SAI: ghi khi cửa sổ có thể đã dài k + 1
        if i >= k:
            cnt[nums[i - k]] -= 1                              # bỏ trễ một nhịp
    return ans


print(sai_bo_tre([3, 8, 7, 8, 7, 5], 2, 2), "| đúng:", Solution().findXSum([3, 8, 7, 8, 7, 5], 2, 2))

[11, 15, 23, 22, 15] | đúng: [11, 15, 15, 15, 12]


**Vì sao sai:** tại `i = k` ta ghi kết quả **trước** khi bỏ `nums[0]`, nên cửa sổ lúc ghi có `k + 1` phần tử. **Cách sửa:** bỏ phần tử trái **trước** khi ghi (như code nộp): thêm `nums[i]`, bỏ `nums[i - k]` (nếu `i >= k`), **rồi** mới ghi khi `i >= k - 1`. Cũng nhớ xoá khoá có đếm `0`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Mọi đoạn con liên tiếp độ dài cố định `k`" ⇒ **cửa sổ trượt**: thêm phần tử phải, bỏ phần tử trái; đừng tính lại từ đầu.
- "Hơn nhau theo tiêu chí 1, hoà thì theo tiêu chí 2" ⇒ **tuple** làm khoá so sánh: `(tiêu_chí_1, tiêu_chí_2)`.
- "Lấy `x` phần tử lớn nhất" ⇒ `sorted(...)[:x]` (đơn giản) hoặc `heapq.nlargest(x, ...)` (khi `x` nhỏ so với số phần tử).
- `Counter` + xoá khoá khi đếm về `0` giúp `len(cnt)` luôn là số giá trị phân biệt.
- Muốn kiểm chứng: viết một hàm tham chiếu đọc đúng từng chữ của đề (dựng mảng "giữ lại" thật sự) và so với bản nhanh trên dữ liệu ngẫu nhiên có nhiều trường hợp hoà.
- Trong bài khó hơn (n lớn), nghĩ tới danh sách có thứ tự (`SortedList`) hoặc hai heap với **xoá lười** (lazy deletion).

## 15. Biến thể của bài

1. **Find X-Sum of All K-Long Subarrays II (3321, Hard):** cùng đề nhưng `n` rất lớn (cỡ `10^5`) và giá trị lớn; cần duy trì top `x` theo thời gian `O(log n)` mỗi bước (hai tập có thứ tự). (Ràng buộc chi tiết: đối chiếu lại đề gốc.)
2. **Top K Frequent Elements (347)**, **Top K Frequent Words (692):** cùng ý "xếp theo `(tần suất, tiêu chí phụ)`".
3. **Maximum Average Subarray I (643):** cửa sổ trượt cơ bản, tổng thay vì tần suất.
4. Luật hoà ngược: **giá trị nhỏ hơn** thắng khi hoà số lần (đổi khoá thành `(số_lần, -giá_trị)`).
5. Đếm **số giá trị phân biệt** trong mỗi cửa sổ (`len(cnt)` khi duy trì đếm).

In [28]:
def x_sum_small_wins(nums, k, x):
    # luật hoà ngược: hoà số lần thì giá trị NHỎ hơn thắng
    ans = []
    for i in range(len(nums) - k + 1):
        cnt = Counter(nums[i:i + k])
        top = sorted(((c, -v) for v, c in cnt.items()), reverse=True)[:x]
        ans.append(sum(c * (-nv) for c, nv in top))
    return ans


def distinct_per_window(nums, k):
    cnt = Counter()
    out = []
    for i, v in enumerate(nums):
        cnt[v] += 1
        if i >= k:
            cnt[nums[i - k]] -= 1
            if cnt[nums[i - k]] == 0:
                del cnt[nums[i - k]]
        if i >= k - 1:
            out.append(len(cnt))
    return out


def top_k_frequent(nums, k):
    # biến thể 347 bằng cùng ý tưởng tuple (số_lần, giá_trị)
    cnt = Counter(nums)
    return [v for c, v in heapq.nlargest(k, ((c, v) for v, c in cnt.items()))]


print(x_sum_small_wins([1, 2, 2, 3, 4, 2], 6, 2))        # [7]  (giá trị 2 xuất hiện 3 lần = 6; 1, 3, 4 hoà 1 lần -> chọn 1 nhỏ nhất: 6 + 1 = 7)

[7]


In [29]:
# kiểm tra bằng tay: cửa sổ [1,2,2,3,4,2] x=2: giữ 2 (3 lần) và giá trị nhỏ nhất trong nhóm hoà 1 lần là 1
assert x_sum_small_wins([1, 2, 2, 3, 4, 2], 6, 2) == [7]
print(distinct_per_window([1, 1, 2, 2, 3, 4, 2, 3], 6))  # [4, 4, 3]
assert distinct_per_window([1, 1, 2, 2, 3, 4, 2, 3], 6) == [4, 4, 3]
print(top_k_frequent([1, 1, 1, 2, 2, 3], 2))             # [1, 2]
assert top_k_frequent([1, 1, 1, 2, 2, 3], 2) == [1, 2]

[4, 4, 3]
[1, 2]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Tính lại mỗi cửa sổ | `Counter` + sắp xếp cho từng đoạn | `O(n·(k + d log d))` | `O(k)` | hiểu đề, đối chiếu |
| 2. Trượt + sắp xếp | cập nhật đếm `O(1)`, sắp xếp `d` cặp | `O(n·d log d)` | `O(d)` | **nên nộp** (đơn giản, đủ nhanh) |
| 3. Trượt + `nlargest` | chỉ lấy top `x`, không sắp hết | `O(n·d log x)` | `O(d + x)` | `x` nhỏ, `d` lớn |
| 4. Trượt + `bisect` | list có thứ tự, xoá/chèn từng cặp | `O(n·(d + x))` | `O(d)` | bước đệm sang bài II (cấu trúc có thứ tự) |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `Counter("abca")["a"]`, `(2, 5) > (2, 3)`, `sorted([(1, 3), (2, 1)], reverse=True)`, số cửa sổ khi `n = 8, k = 6`, và `Solution().findXSum([4, 4], 2, 1)`.

**Bài 2 (tính tay).** Với `nums = [1, 2, 2, 3, 3, 3]`, `k = 4`, `x = 1`: lập bảng cho từng cửa sổ (đếm, cặp xếp giảm dần, top 1, x-sum).

**Bài 3 (điền TODO).** Hoàn thành `x_sum_of(window, x)` — tính x-sum của **một** mảng.

In [30]:
def x_sum_of(window, x):
    # TODO 1: cnt = Counter(window)
    # TODO 2: tạo list pairs gồm (số_lần, giá_trị), sắp xếp giảm dần
    # TODO 3: lấy x cặp đầu, trả về tổng số_lần * giá_trị
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `find_xsum_nlargest(nums, k, x)` theo Cách 3 (`heapq.nlargest`) **mà không nhìn code mẫu**; nhớ bỏ phần tử trái **trước** khi ghi kết quả.

**Bài 5 (biến thể / mini challenge).** (a) Viết `x_sum_small_wins(nums, k, x)` với luật hoà ngược (giá trị **nhỏ hơn** thắng). (b) Viết `kept_values_per_window(nums, k, x)` trả về, cho mỗi cửa sổ, **tập các giá trị được giữ** (dạng list tăng dần).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `2`, `True`, `[(2, 1), (1, 3)]`, `3`, `8` (một giá trị `4` xuất hiện 2 lần, `x = 1`). Bài 3: bắt đầu bằng `Counter(window)`; tuple là `(c, v)`.
- **Mức 2 (vừa):** Bài 2: cửa sổ `[1,2,2,3]` → `(2,2)` thắng → `4`; `[2,2,3,3]` hoà `(2,2)` và `(2,3)` → chọn giá trị lớn `3` → `6`; `[2,3,3,3]` → `(3,3)` → `9`. Bài 4: giống Cách 2, thay dòng tính tổng bằng `heapq.nlargest`.
- **Mức 3 (sâu):** Bài 5(a): dùng khoá `(số_lần, -giá_trị)` rồi đảo dấu lại khi nhân. Bài 5(b): sau khi lấy top, thu `v` của từng cặp rồi `sorted`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert Counter("abca")["a"] == 2
assert (2, 5) > (2, 3)
assert sorted([(1, 3), (2, 1)], reverse=True) == [(2, 1), (1, 3)]
assert 8 - 6 + 1 == 3
assert Solution().findXSum([4, 4], 2, 1) == [8]

# Bài 2
assert Solution().findXSum([1, 2, 2, 3, 3, 3], 4, 1) == [4, 6, 9]


# Bài 3
def x_sum_of_ans(window, x):
    cnt = Counter(window)
    pairs = sorted(((c, v) for v, c in cnt.items()), reverse=True)
    return sum(c * v for c, v in pairs[:x])


assert x_sum_of_ans([1, 1, 2, 2, 3, 4], 2) == 6
assert x_sum_of_ans([1, 2, 2, 3, 4, 2], 2) == 10
assert x_sum_of_ans([5, 5, 5], 4) == 15


# Bài 4
def find_xsum_nlargest(nums, k, x):
    cnt = Counter()
    ans = []
    for i, v in enumerate(nums):
        cnt[v] += 1
        if i >= k:
            cnt[nums[i - k]] -= 1
            if cnt[nums[i - k]] == 0:
                del cnt[nums[i - k]]
        if i >= k - 1:
            top = heapq.nlargest(x, ((c, val) for val, c in cnt.items()))
            ans.append(sum(c * val for c, val in top))
    return ans


for (nums, k, x), expected in examples:
    assert find_xsum_nlargest(nums, k, x) == expected


# Bài 5(a)
def x_sum_small_wins_ans(nums, k, x):
    ans = []
    for i in range(len(nums) - k + 1):
        cnt = Counter(nums[i:i + k])
        top = sorted(((c, -v) for v, c in cnt.items()), reverse=True)[:x]
        ans.append(sum(c * -nv for c, nv in top))
    return ans


assert x_sum_small_wins_ans([1, 2, 2, 3, 4, 2], 6, 2) == [7]
assert x_sum_small_wins_ans([1, 1, 2, 2, 3, 4], 6, 2) == [6]     # 1 và 2 cùng 2 lần, cả hai được giữ


# Bài 5(b)
def kept_values_per_window(nums, k, x):
    out = []
    for i in range(len(nums) - k + 1):
        cnt = Counter(nums[i:i + k])
        top = sorted(((c, v) for v, c in cnt.items()), reverse=True)[:x]
        out.append(sorted(v for c, v in top))
    return out


assert kept_values_per_window([1, 1, 2, 2, 3, 4, 2, 3], 6, 2) == [[1, 2], [2, 4], [2, 3]]
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (cửa sổ trượt + đếm + top `x` theo `(số_lần, giá_trị)`)

```python
class Solution:
    def findXSum(self, nums: List[int], k: int, x: int) -> List[int]:
        cnt = Counter()
        ans = []
        for i, v in enumerate(nums):
            cnt[v] += 1                       # thêm phải
            if i >= k:                        # bỏ trái trước khi ghi
                cnt[nums[i - k]] -= 1
                if cnt[nums[i - k]] == 0:
                    del cnt[nums[i - k]]
            if i >= k - 1:                    # đủ k phần tử
                pairs = sorted(((c, val) for val, c in cnt.items()), reverse=True)
                ans.append(sum(c * val for c, val in pairs[:x]))
        return ans
```

**Checklist trước khi submit**

- [ ] Khoá so sánh là `(số_lần, giá_trị)` giảm dần (hoà thì giá trị **lớn** hơn thắng)?
- [ ] Tổng là `số_lần × giá_trị`, không phải chỉ `giá_trị`?
- [ ] Bỏ phần tử trái **trước** khi ghi kết quả, và xoá khoá khi đếm về `0`?
- [ ] Có đúng `n - k + 1` kết quả?
- [ ] Trường hợp ít hơn `x` giá trị phân biệt: `pairs[:x]` tự lấy hết?
- [ ] Đã thử ví dụ 1, ví dụ 2, `k = n`, `k = 1`, `x = k` và mảng toàn phần tử giống nhau?